
# 07 Final Reviewer Runs — no rerun of notebooks 1–6

This notebook is the fixed version.

It **does not assume notebooks 1–6 produce prediction CSVs**. Those notebooks mainly create preprocessing artifacts like masks, detector features, and evaluation windows. The old `07` failed because it searched for per-window prediction CSVs that your pipeline never saved.

Run this notebook from the **repo root**, the same folder that contains `data/`.

What it does:

1. Verifies/rebuilds required data artifacts if possible.
2. Collects diagnostics from `data/`.
3. Builds final paper tables/plots from:
   - exact aggregate paper numbers already reported, and
   - any prediction CSVs / model outputs if you actually have them.
4. Generates reviewer-facing outputs in `final_reviewer_outputs/`.
5. Never crashes just because prediction CSVs are missing.

Important: if this notebook says `aggregate-only fallback`, then it is good for paper/table regeneration, but not a full model rerun. A full model rerun requires the actual training/evaluation notebook or saved per-window predictions.


In [ ]:

# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import os, sys, re, json, math, glob, shutil, subprocess, warnings, ast
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

OUT = ROOT / "final_reviewer_outputs"
FIG_DIR = OUT / "figures"
TABLE_DIR = OUT / "tables"
LOG_DIR = OUT / "logs"
for p in [OUT, FIG_DIR, TABLE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# This fixed notebook does NOT rerun preprocessing notebooks.
RUN_ANY_NOTEBOOKS = False

# If you have an experiments notebook that actually trains/evaluates models, list it here.
# Example: EXPERIMENT_NOTEBOOKS = ["experiments.ipynb"]
EXPERIMENT_NOTEBOOKS = []

# If you later save exact per-window predictions, add them here.
# Best format: dataset, method, task/horizon, window_id, y_true, y_pred.
FORCE_PREDICTION_FILES = []

# Search paths for optional existing prediction/result CSVs.
PREDICTION_CSV_GLOBS = [
    "results/**/*.csv", "outputs/**/*.csv", "figures/**/*.csv", "tables/**/*.csv",
    "eval/**/*.csv", "evaluation/**/*.csv", "runs/**/*.csv", "notebook_outputs/**/*.csv",
    "*.csv"
]

BOOTSTRAP_B = 5000
BOOTSTRAP_SEED = 123

PAPER_METHOD_ORDER = ["LOCF", "LinearInterp + SeasonalNaive", "MAR (LDS)", "MNAR (LDS)"]
HORIZON_ORDER = ["impute", "1-step", "3-step", "6-step"]
LENGTH_BINS = [0, 6, 12, 24, 72, np.inf]
LENGTH_LABELS = ["1-6", "7-12", "13-24", "25-72", "73+"]

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("Outputs:", OUT)


In [ ]:

# ============================================================
# 0) Verify data artifacts
# ============================================================
required = [
    "x_t_nan.npy",
    "m_t.npy",
    "detector_ids.npy",
    "time_features.npy",
    "detector_features.npy",
    "evaluation_windows.parquet",
    "blackout_events_detectors.parquet",
    "blackout_events_network.parquet",
]

print("Data artifact check:")
for f in required:
    p = DATA_DIR / f
    print(f"{f:40s}", "OK" if p.exists() else "MISSING")

# If x_t_nan.npy is missing but seattle_loop_clean.pkl exists, rebuild it.
if not (DATA_DIR / "x_t_nan.npy").exists() and (DATA_DIR / "seattle_loop_clean.pkl").exists():
    panel = pd.read_pickle(DATA_DIR / "seattle_loop_clean.pkl")
    X = panel.to_numpy(dtype=float)
    np.save(DATA_DIR / "x_t_nan.npy", X)
    np.save(DATA_DIR / "m_t.npy", np.isnan(X).astype(np.int8))
    np.save(DATA_DIR / "detector_ids.npy", panel.columns.astype(str).to_numpy())
    print("Rebuilt x_t_nan.npy, m_t.npy, detector_ids.npy from seattle_loop_clean.pkl")

# If seattle_loop_clean.pkl is missing but speed_matrix_2015 exists, rebuild core files.
speed_candidates = [
    DATA_DIR / "speed_matrix_2015",
    DATA_DIR / "speed_matrix_2015.pkl",
    ROOT / "speed_matrix_2015",
    ROOT / "speed_matrix_2015.pkl",
]
if not (DATA_DIR / "x_t_nan.npy").exists():
    speed_path = next((p for p in speed_candidates if p.exists()), None)
    if speed_path is not None:
        print("Recovering from:", speed_path)
        panel = pd.read_pickle(speed_path)
        panel.index = pd.to_datetime(panel.index)
        panel = panel.sort_index()
        panel.columns = panel.columns.astype(str)
        full_index = pd.date_range("2015-01-01 00:00:00", "2015-12-31 23:55:00", freq="5min")
        panel = panel.reindex(full_index)
        panel = panel.replace(0, np.nan)
        ids_path = DATA_DIR / "detector_ids.npy"
        if ids_path.exists():
            old_ids = np.load(ids_path, allow_pickle=True).astype(str)
            keep = [d for d in old_ids if d in panel.columns]
            print("Matched old detector IDs:", len(keep), "of", len(old_ids))
            if len(keep) >= 100:
                panel = panel[keep]
            else:
                keep = panel.isna().mean().sort_values().index[:147]
                panel = panel[keep]
        else:
            keep = panel.isna().mean().sort_values().index[:147]
            panel = panel[keep]
        panel.to_pickle(DATA_DIR / "seattle_loop_clean.pkl")
        try:
            panel.to_parquet(DATA_DIR / "seattle_loop_clean.parquet")
        except Exception as e:
            print("Could not save parquet; continuing:", e)
        X = panel.to_numpy(dtype=float)
        np.save(DATA_DIR / "x_t_nan.npy", X)
        np.save(DATA_DIR / "m_t.npy", np.isnan(X).astype(np.int8))
        np.save(DATA_DIR / "detector_ids.npy", panel.columns.astype(str).to_numpy())
        print("Recovered core arrays. Shape:", X.shape, "missing fraction:", np.isnan(X).mean())

print("\nShapes:")
if (DATA_DIR / "x_t_nan.npy").exists():
    X = np.load(DATA_DIR / "x_t_nan.npy")
    print("x_t_nan:", X.shape, "missing frac:", np.isnan(X).mean())
if (DATA_DIR / "m_t.npy").exists():
    m_t = np.load(DATA_DIR / "m_t.npy")
    print("m_t:", m_t.shape)
if (DATA_DIR / "detector_ids.npy").exists():
    ids = np.load(DATA_DIR / "detector_ids.npy", allow_pickle=True)
    print("detector_ids:", ids.shape)
if (DATA_DIR / "evaluation_windows.parquet").exists():
    ew = pd.read_parquet(DATA_DIR / "evaluation_windows.parquet")
    print("evaluation_windows:", ew.shape)
    print("evaluation columns:", list(ew.columns))
    display(ew.head())


In [ ]:

# ============================================================
# Optional: run experiments notebooks only, not preprocessing notebooks
# ============================================================
if RUN_ANY_NOTEBOOKS and EXPERIMENT_NOTEBOOKS:
    for nb in EXPERIMENT_NOTEBOOKS:
        nb_path = ROOT / nb
        if not nb_path.exists():
            print("Missing notebook:", nb_path)
            continue
        out_name = nb_path.stem + "__executed.ipynb"
        log_path = LOG_DIR / f"{nb_path.stem}.log"
        cmd = [
            sys.executable, "-m", "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute", str(nb_path),
            "--ExecutePreprocessor.timeout=-1",
            f"--ExecutePreprocessor.cwd={ROOT}",
            "--output", out_name,
            "--output-dir", str(OUT),
        ]
        print("Running:", nb)
        with open(log_path, "w", encoding="utf-8") as logf:
            proc = subprocess.run(cmd, cwd=ROOT, stdout=logf, stderr=subprocess.STDOUT, text=True)
        if proc.returncode != 0:
            raise RuntimeError(f"Notebook failed: {nb}. See log: {log_path}")
        print("Done:", nb)
else:
    print("Skipping notebook execution. This notebook will use existing files + aggregate fallback.")


In [ ]:

# ============================================================
# Utilities
# ============================================================
def clean_method(x):
    if pd.isna(x): return x
    s = str(x).strip()
    low = s.lower().replace("_", " ").replace("-", " ")
    if low in {"locf", "last observation carried forward", "last carried forward"} or "locf" in low:
        return "LOCF"
    if "linear" in low and ("season" in low or "naive" in low):
        return "LinearInterp + SeasonalNaive"
    if "mnar" in low:
        return "MNAR (LDS)"
    if "mar" in low and "mnar" not in low:
        return "MAR (LDS)"
    if low in {"lds", "kalman", "kf"}:
        return "MAR (LDS)"
    return s

def clean_dataset(x, source=""):
    text = (str(x) if not pd.isna(x) else "") + " " + str(source)
    low = text.lower()
    if "metr" in low:
        return "METR-LA"
    if "synthetic" in low or "alpha" in low:
        return "Synthetic"
    if "seattle" in low or "loop" in low:
        return "Seattle"
    return str(x).strip() if not pd.isna(x) and str(x).strip() else "Unknown"

def clean_task_horizon(task=None, horizon=None, source=""):
    text = " ".join([str(task) if task is not None else "", str(horizon) if horizon is not None else "", str(source)])
    low = text.lower()
    if "imput" in low or "recon" in low:
        return "impute"
    for h in [1, 3, 6]:
        pats = [f"{h}-step", f"{h} step", f"h{h}", f"h={h}", f"horizon_{h}", f"horizon {h}", f"{h}step"]
        if any(p in low for p in pats):
            return f"{h}-step"
    try:
        hv = int(float(str(horizon)))
        if hv in [1,3,6]: return f"{hv}-step"
    except Exception:
        pass
    if "forecast" in low or "pred" in low:
        return "forecast"
    return str(task).strip() if task is not None and not pd.isna(task) and str(task).strip() else "unknown"

def first_col(df, candidates):
    lower_to_col = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns: return c
        if c.lower() in lower_to_col: return lower_to_col[c.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def parse_numeric_array(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        arr = np.asarray(v, dtype=float).ravel()
        return arr[np.isfinite(arr)]
    if pd.isna(v):
        return np.array([], dtype=float)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return np.array([float(v)], dtype=float)
    s = str(v).strip()
    if not s:
        return np.array([], dtype=float)
    for loader in (json.loads, ast.literal_eval):
        try:
            obj = loader(s)
            arr = np.asarray(obj, dtype=float).ravel()
            return arr[np.isfinite(arr)]
        except Exception:
            pass
    nums = re.findall(r"[-+]?\d*\.\d+(?:[eE][-+]?\d+)?|[-+]?\d+(?:[eE][-+]?\d+)?", s)
    if nums:
        arr = np.asarray([float(n) for n in nums], dtype=float)
        return arr[np.isfinite(arr)]
    return np.array([], dtype=float)

def row_sqerr_sum_n(y_true, y_pred):
    yt = parse_numeric_array(y_true)
    yp = parse_numeric_array(y_pred)
    n = min(len(yt), len(yp))
    if n == 0:
        return np.nan, 0
    diff = yp[:n] - yt[:n]
    return float(np.sum(diff * diff)), int(n)

def rmse_from_sqerr(df):
    n = df["n_obs"].sum()
    if n <= 0:
        return np.nan
    return float(np.sqrt(df["sqerr_sum"].sum() / n))

def bootstrap_rmse_ci(df, B=5000, seed=123, id_col="window_id"):
    rng = np.random.default_rng(seed)
    tmp = df.copy()
    if id_col not in tmp.columns:
        tmp[id_col] = np.arange(len(tmp))
    ids = tmp[id_col].dropna().unique()
    if len(ids) == 0:
        return np.nan, np.nan, np.nan
    grouped = {k: g for k, g in tmp.groupby(id_col)}
    vals = []
    for _ in range(B):
        sample_ids = rng.choice(ids, size=len(ids), replace=True)
        sq = 0.0; n = 0.0
        for sid in sample_ids:
            g = grouped[sid]
            sq += g["sqerr_sum"].sum()
            n += g["n_obs"].sum()
        vals.append(np.sqrt(sq/n) if n > 0 else np.nan)
    vals = np.asarray(vals)
    vals = vals[np.isfinite(vals)]
    return rmse_from_sqerr(tmp), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def paired_delta_bootstrap(df, method_a="MNAR (LDS)", method_b="MAR (LDS)", B=5000, seed=123):
    d = df[df["method"].isin([method_a, method_b])].copy()
    if d.empty or "window_id" not in d.columns:
        return None
    agg = d.groupby(["method", "window_id"], as_index=False).agg(sqerr_sum=("sqerr_sum", "sum"), n_obs=("n_obs", "sum"))
    pivot_sq = agg.pivot(index="window_id", columns="method", values="sqerr_sum")
    pivot_n = agg.pivot(index="window_id", columns="method", values="n_obs")
    common = pivot_sq.dropna(subset=[method_a, method_b]).index.intersection(
        pivot_n.dropna(subset=[method_a, method_b]).index
    )
    if len(common) == 0:
        return None
    sq_a = pivot_sq.loc[common, method_a].to_numpy(float)
    sq_b = pivot_sq.loc[common, method_b].to_numpy(float)
    n_a = pivot_n.loc[common, method_a].to_numpy(float)
    n_b = pivot_n.loc[common, method_b].to_numpy(float)
    def delta(ii):
        return np.sqrt(sq_a[ii].sum()/n_a[ii].sum()) - np.sqrt(sq_b[ii].sum()/n_b[ii].sum())
    idx = np.arange(len(common))
    point = delta(idx)
    rng = np.random.default_rng(seed)
    vals = np.array([delta(rng.choice(idx, size=len(idx), replace=True)) for _ in range(B)])
    return {
        "n_windows": int(len(common)),
        "delta_rmse": float(point),
        "ci_low": float(np.percentile(vals, 2.5)),
        "ci_high": float(np.percentile(vals, 97.5)),
    }

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

print("Utilities loaded.")


In [ ]:

# ============================================================
# 1) Try to load actual per-window predictions/results if they exist
# ============================================================
def candidate_csv_files():
    files = []
    files.extend([ROOT / f for f in FORCE_PREDICTION_FILES])
    for pat in PREDICTION_CSV_GLOBS:
        files.extend(ROOT.glob(pat))
    clean = []
    seen = set()
    for f in files:
        if not f.is_file():
            continue
        if OUT in f.parents:
            continue
        if f.resolve() in seen:
            continue
        seen.add(f.resolve())
        clean.append(f)
    return sorted(clean)

def standardize_prediction_csv(path):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if df.empty:
        return None

    true_col = first_col(df, ["y_true", "true", "actual", "target", "gt", "ground_truth", "x_true"])
    pred_col = first_col(df, ["y_pred", "pred", "prediction", "forecast", "imputed", "x_pred", "x_hat"])
    err_col = first_col(df, ["sqerr_sum", "squared_error_sum", "se_sum"])
    n_col = first_col(df, ["n_obs", "n", "count", "num_obs"])
    rmse_col = first_col(df, ["rmse"])
    method_col = first_col(df, ["method", "model", "model_name", "approach"])

    # Need predictions/errors/rmse to be usable.
    if true_col is None and pred_col is None and err_col is None and rmse_col is None:
        return None

    out = pd.DataFrame()
    out["source_file"] = str(path.relative_to(ROOT))
    out["method"] = df[method_col].map(clean_method) if method_col else clean_method(path.stem)

    dataset_col = first_col(df, ["dataset", "data", "city"])
    out["dataset"] = df[dataset_col].map(lambda x: clean_dataset(x, path)) if dataset_col else clean_dataset(None, path)

    task_col = first_col(df, ["task", "eval_task", "split_task", "type"])
    horizon_col = first_col(df, ["horizon", "h", "step", "forecast_horizon"])
    task_vals = df[task_col] if task_col else pd.Series([None]*len(df))
    hor_vals = df[horizon_col] if horizon_col else pd.Series([None]*len(df))
    out["task_horizon"] = [clean_task_horizon(t, h, path) for t, h in zip(task_vals, hor_vals)]

    win_col = first_col(df, ["window_id", "window", "event_id", "blackout_id", "id"])
    out["window_id"] = df[win_col].astype(str) if win_col else [f"{path.stem}_row{i}" for i in range(len(df))]

    len_col = first_col(df, ["blackout_len", "length", "len", "duration", "blackout_length"])
    out["blackout_len"] = pd.to_numeric(df[len_col], errors="coerce") if len_col else np.nan

    time_col = first_col(df, ["start_time", "timestamp", "time", "start", "blackout_start"])
    out["start_time"] = pd.to_datetime(df[time_col], errors="coerce") if time_col else pd.NaT

    seed_col = first_col(df, ["seed", "random_seed", "run_seed"])
    out["seed"] = pd.to_numeric(df[seed_col], errors="coerce") if seed_col else np.nan

    alpha_col = first_col(df, ["alpha", "mnar_alpha", "strength"])
    out["alpha"] = pd.to_numeric(df[alpha_col], errors="coerce") if alpha_col else np.nan

    if err_col and n_col:
        out["sqerr_sum"] = pd.to_numeric(df[err_col], errors="coerce")
        out["n_obs"] = pd.to_numeric(df[n_col], errors="coerce").fillna(1)
        out["aggregate_only"] = False
    elif true_col and pred_col:
        vals = [row_sqerr_sum_n(t, p) for t, p in zip(df[true_col], df[pred_col])]
        out["sqerr_sum"] = [v[0] for v in vals]
        out["n_obs"] = [v[1] for v in vals]
        out["aggregate_only"] = False
    elif rmse_col:
        # Aggregate fallback from a CSV. Not valid for paired bootstrap.
        r = pd.to_numeric(df[rmse_col], errors="coerce")
        out["sqerr_sum"] = r ** 2
        out["n_obs"] = 1.0
        out["aggregate_only"] = True
    else:
        return None

    out = out[(out["n_obs"] > 0) & np.isfinite(out["sqerr_sum"])]
    return out if not out.empty else None

frames = []
for f in candidate_csv_files():
    sdf = standardize_prediction_csv(f)
    if sdf is not None and not sdf.empty:
        frames.append(sdf)
        print("Using CSV:", f.relative_to(ROOT), "rows=", len(sdf))

if frames:
    pred = pd.concat(frames, ignore_index=True)
    pred["method"] = pred["method"].map(clean_method)
    pred["dataset"] = [clean_dataset(d, s) for d, s in zip(pred["dataset"], pred["source_file"])]
    pred["task_horizon"] = [clean_task_horizon(th, None, s) for th, s in zip(pred["task_horizon"], pred["source_file"])]
    pred.to_csv(TABLE_DIR / "standardized_predictions_all.csv", index=False)
    print("Loaded actual/CSV prediction rows:", len(pred))
else:
    pred = pd.DataFrame()
    print("No usable prediction CSVs found. This is okay: falling back to aggregate paper numbers.")


In [ ]:

# ============================================================
# 2) Aggregate fallback: exact paper numbers from the current draft
# ============================================================
# These reproduce the tables/figures even if per-window predictions were not saved.
# If pred above is nonempty, the notebook will prefer pred for RMSE tables and bootstrap.

aggregate_rows = [
    # Seattle
    ("Seattle", "LOCF", "impute", 7.021),
    ("Seattle", "LOCF", "1-step", 7.860),
    ("Seattle", "LOCF", "3-step", 8.465),
    ("Seattle", "LOCF", "6-step", 8.942),
    ("Seattle", "LinearInterp + SeasonalNaive", "impute", 5.024),
    ("Seattle", "LinearInterp + SeasonalNaive", "1-step", 8.734),
    ("Seattle", "LinearInterp + SeasonalNaive", "3-step", 8.442),
    ("Seattle", "LinearInterp + SeasonalNaive", "6-step", 8.881),
    ("Seattle", "MAR (LDS)", "impute", 4.229),
    ("Seattle", "MAR (LDS)", "1-step", 4.391),
    ("Seattle", "MAR (LDS)", "3-step", 4.349),
    ("Seattle", "MAR (LDS)", "6-step", 5.160),
    ("Seattle", "MNAR (LDS)", "impute", 4.195),
    ("Seattle", "MNAR (LDS)", "1-step", 4.313),
    ("Seattle", "MNAR (LDS)", "3-step", 4.228),
    ("Seattle", "MNAR (LDS)", "6-step", 5.104),

    # METR-LA
    ("METR-LA", "LOCF", "impute", 9.433),
    ("METR-LA", "LOCF", "1-step", 9.855),
    ("METR-LA", "LOCF", "3-step", 9.262),
    ("METR-LA", "LOCF", "6-step", 9.147),
    ("METR-LA", "MAR (LDS)", "impute", 5.483),
    ("METR-LA", "MAR (LDS)", "1-step", 4.700),
    ("METR-LA", "MAR (LDS)", "3-step", 5.320),
    ("METR-LA", "MAR (LDS)", "6-step", 5.114),
    ("METR-LA", "MNAR (LDS)", "impute", 5.355),
    ("METR-LA", "MNAR (LDS)", "1-step", 4.700),
    ("METR-LA", "MNAR (LDS)", "3-step", 5.238),
    ("METR-LA", "MNAR (LDS)", "6-step", 4.790),
]
aggregate = pd.DataFrame(aggregate_rows, columns=["dataset", "method", "task_horizon", "rmse"])
aggregate["aggregate_only"] = True
aggregate.to_csv(TABLE_DIR / "aggregate_paper_numbers_from_draft.csv", index=False)
print("Saved aggregate paper numbers:", TABLE_DIR / "aggregate_paper_numbers_from_draft.csv")
display(aggregate.head())


In [ ]:

# ============================================================
# 3) Main RMSE tables
# ============================================================
if not pred.empty:
    rows = []
    for (dataset, task_horizon, method), g in pred.groupby(["dataset", "task_horizon", "method"]):
        rows.append({
            "dataset": dataset,
            "task_horizon": task_horizon,
            "method": method,
            "rmse": rmse_from_sqerr(g),
            "n_rows": len(g),
            "n_obs": int(g["n_obs"].sum()),
            "n_windows": g["window_id"].nunique(),
            "aggregate_only": bool(g["aggregate_only"].any()),
        })
    main = pd.DataFrame(rows)
    source_mode = "prediction_csvs"
else:
    main = aggregate.copy()
    main["n_rows"] = np.nan
    main["n_obs"] = np.nan
    main["n_windows"] = np.nan
    source_mode = "aggregate_only_fallback"

main["method_order"] = main["method"].map({m:i for i,m in enumerate(PAPER_METHOD_ORDER)}).fillna(999)
main["horizon_order"] = main["task_horizon"].map({h:i for i,h in enumerate(HORIZON_ORDER)}).fillna(999)
main = main.sort_values(["dataset", "horizon_order", "method_order", "method"])
main.to_csv(TABLE_DIR / "main_rmse_long.csv", index=False)

print("RMSE source mode:", source_mode)
print("Saved:", TABLE_DIR / "main_rmse_long.csv")

for dataset in main["dataset"].unique():
    piv = main[main["dataset"]==dataset].pivot_table(index="method", columns="task_horizon", values="rmse", aggfunc="first")
    cols = [c for c in HORIZON_ORDER if c in piv.columns] + [c for c in piv.columns if c not in HORIZON_ORDER]
    idx = [m for m in PAPER_METHOD_ORDER if m in piv.index] + [m for m in piv.index if m not in PAPER_METHOD_ORDER]
    piv = piv.loc[idx, cols]
    safe = dataset.lower().replace("-", "").replace(" ", "_")
    piv.to_csv(TABLE_DIR / f"{safe}_rmse_wide.csv")
    print("\n===", dataset, "===")
    display(piv.round(3))


In [ ]:

# ============================================================
# 4) Bootstrap / uncertainty
# ============================================================
boot = pd.DataFrame()

if not pred.empty and not pred.get("aggregate_only", pd.Series([True])).all():
    boot_rows = []
    for (dataset, task_horizon), g in pred.groupby(["dataset", "task_horizon"]):
        res = paired_delta_bootstrap(g, "MNAR (LDS)", "MAR (LDS)", B=BOOTSTRAP_B, seed=BOOTSTRAP_SEED)
        if res is not None:
            boot_rows.append({"dataset": dataset, "task_horizon": task_horizon, **res})
    boot = pd.DataFrame(boot_rows)

if boot.empty:
    # Known paired-window imputation result from earlier exact run.
    # Keep this only if it matches your final aligned-window evaluation.
    boot = pd.DataFrame([
        {
            "dataset": "Seattle",
            "task_horizon": "impute",
            "n_windows": np.nan,
            "delta_rmse": -0.0584,
            "ci_low": -0.1079,
            "ci_high": -0.0150,
            "note": "fallback_known_paired_window_bootstrap_from_earlier_run"
        }
    ])
    print("WARNING: Using fallback known paired-window bootstrap for Seattle imputation.")
    print("Do not use this row if your final aligned-window run changed.")

boot["horizon_order"] = boot["task_horizon"].map({h:i for i,h in enumerate(HORIZON_ORDER)}).fillna(999)
boot = boot.sort_values(["dataset", "horizon_order"])
boot.to_csv(TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv", index=False)
print("Saved:", TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv")
display(boot.drop(columns=["horizon_order"]).round(4))


In [ ]:

# ============================================================
# 5) Reviewer figures from aggregate numbers or prediction CSVs
# ============================================================

# 5.1 Seattle imputation bar plot
sea_imp_main = main[(main["dataset"]=="Seattle") & (main["task_horizon"]=="impute")].copy()
if not sea_imp_main.empty:
    sea_imp_main["order"] = sea_imp_main["method"].map({m:i for i,m in enumerate(PAPER_METHOD_ORDER)}).fillna(999)
    sea_imp_main = sea_imp_main.sort_values(["order", "method"])
    x = np.arange(len(sea_imp_main))
    y = sea_imp_main["rmse"].to_numpy(float)

    plt.figure(figsize=(7.2, 4.2))
    plt.bar(x, y)
    plt.xticks(x, sea_imp_main["method"], rotation=20, ha="right")
    plt.ylabel("RMSE (mph)")
    title = "Seattle Loop imputation RMSE"
    if source_mode == "aggregate_only_fallback":
        title += " (aggregate paper numbers)"
    plt.title(title)
    savefig(FIG_DIR / "impute_rmse_methods_ci_labels.png")

# 5.2 Forecast horizon plot
sea_fc = main[(main["dataset"]=="Seattle") & (main["task_horizon"].isin(["1-step","3-step","6-step"]))].copy()
if not sea_fc.empty:
    sea_fc["h"] = sea_fc["task_horizon"].str.extract(r"(\d+)").astype(float)
    plt.figure(figsize=(7.2, 4.2))
    for method in [m for m in PAPER_METHOD_ORDER if m in sea_fc["method"].unique()]:
        gm = sea_fc[sea_fc["method"]==method].sort_values("h")
        plt.plot(gm["h"], gm["rmse"], marker="o", label=method)
    plt.xticks([1,3,6], ["1-step\n5 min", "3-step\n15 min", "6-step\n30 min"])
    plt.ylabel("RMSE (mph)")
    plt.xlabel("Forecast horizon")
    title = "Seattle Loop post-blackout forecasting RMSE"
    if source_mode == "aggregate_only_fallback":
        title += " (aggregate paper numbers)"
    plt.title(title)
    plt.legend(frameon=False)
    savefig(FIG_DIR / "forecast_rmse_by_horizon_panels_ci.png")

# 5.3 Synthetic alpha sweep
alpha_df = pd.DataFrame({
    "alpha": [0.0, 0.4, 0.8, 1.2],
    "mar_mean": [3.741, 3.858, 3.750, 3.849],
    "mar_std": [0.026, 0.082, 0.445, 0.510],
    "mnar_mean": [3.730, 3.630, 3.698, 3.659],
    "mnar_std": [0.023, 0.063, 0.538, 0.946],
    "delta_mean": [-0.011, -0.227, -0.052, -0.190],
    "delta_std": [0.007, 0.033, 0.109, 0.372],
})
alpha_df.to_csv(TABLE_DIR / "synthetic_alpha_sweep_delta.csv", index=False)

plt.figure(figsize=(6.4, 4.0))
plt.axhline(0, linestyle="--", linewidth=1)
plt.errorbar(alpha_df["alpha"], alpha_df["delta_mean"], yerr=alpha_df["delta_std"], marker="o", capsize=4)
plt.xlabel(r"State-dependence strength $\alpha$")
plt.ylabel(r"$\Delta$RMSE = MNAR - MAR")
plt.title("Synthetic MNAR-strength sweep")
savefig(FIG_DIR / "synthetic_validation_alpha_sweep.png")


In [ ]:

# ============================================================
# 6) Diagnostics from evaluation windows and blackout parquet files
# ============================================================
# These do not require model predictions.

# Evaluation-window length bucket counts
if (DATA_DIR / "evaluation_windows.parquet").exists():
    ew = pd.read_parquet(DATA_DIR / "evaluation_windows.parquet")
    len_col = first_col(ew, ["blackout_len", "length", "len", "duration", "blackout_length"])
    task_col = first_col(ew, ["task", "type", "eval_task"])
    win_col = first_col(ew, ["window_id", "window", "id", "event_id"])
    if len_col:
        tmp = ew.copy()
        # Keep impute rows if a task column exists; otherwise use all unique windows.
        if task_col:
            mask = tmp[task_col].astype(str).str.lower().str.contains("imput|recon", regex=True)
            if mask.any():
                tmp = tmp[mask]
        tmp["blackout_len"] = pd.to_numeric(tmp[len_col], errors="coerce")
        tmp["length_bucket"] = pd.cut(tmp["blackout_len"], bins=LENGTH_BINS, labels=LENGTH_LABELS, include_lowest=True)
        if win_col:
            counts = tmp.groupby("length_bucket", observed=True)[win_col].nunique().rename("n_windows").reset_index()
        else:
            counts = tmp.groupby("length_bucket", observed=True).size().rename("n_windows").reset_index()
        counts.to_csv(TABLE_DIR / "evaluation_window_length_bucket_counts.csv", index=False)
        print("Evaluation-window length buckets:")
        display(counts)

# Blackout event summaries
for fname in ["blackout_events_detectors.parquet", "blackout_events_network.parquet"]:
    p = DATA_DIR / fname
    if p.exists():
        df = pd.read_parquet(p)
        print("\n", fname, df.shape)
        display(df.head())
        # Save simple summary.
        summary = df.describe(include="all").transpose()
        summary.to_csv(TABLE_DIR / fname.replace(".parquet", "_describe.csv"))

# Missingness fraction
if (DATA_DIR / "x_t_nan.npy").exists():
    X = np.load(DATA_DIR / "x_t_nan.npy")
    miss = np.isnan(X).mean()
    pd.DataFrame([{"missing_fraction": miss, "T": X.shape[0], "D": X.shape[1]}]).to_csv(TABLE_DIR / "panel_missingness_summary.csv", index=False)
    print("Panel missing fraction:", miss, "shape:", X.shape)
elif (DATA_DIR / "m_t.npy").exists():
    m = np.load(DATA_DIR / "m_t.npy")
    miss = m.mean()
    pd.DataFrame([{"missing_fraction": miss, "T": m.shape[0], "D": m.shape[1]}]).to_csv(TABLE_DIR / "panel_missingness_summary.csv", index=False)
    print("Mask missing fraction:", miss, "shape:", m.shape)


In [ ]:

# ============================================================
# 7) Seed robustness placeholder/check
# ============================================================
if not pred.empty and "seed" in pred.columns and pred["seed"].notna().any():
    seed_rows = []
    seed_df = pred[pred["seed"].notna()].copy()
    for (dataset, task_horizon, method, seed), g in seed_df.groupby(["dataset", "task_horizon", "method", "seed"]):
        seed_rows.append({
            "dataset": dataset,
            "task_horizon": task_horizon,
            "method": method,
            "seed": int(seed),
            "rmse": rmse_from_sqerr(g),
            "n_windows": g["window_id"].nunique(),
        })
    seed_long = pd.DataFrame(seed_rows)
    seed_long.to_csv(TABLE_DIR / "seed_level_rmse_long.csv", index=False)
    seed_summary = seed_long.groupby(["dataset", "task_horizon", "method"], as_index=False).agg(
        mean_rmse=("rmse","mean"), std_rmse=("rmse","std"), n_seeds=("seed","nunique")
    )
    seed_summary.to_csv(TABLE_DIR / "seed_robustness_summary.csv", index=False)
    display(seed_summary.round(4))
else:
    print("No seed-level predictions found.")
    print("Paper-safe wording: only claim seed robustness if seed_robustness_summary.csv exists from real multi-seed outputs.")


In [ ]:

# ============================================================
# 8) Final summary markdown + LaTeX snippets
# ============================================================
summary_lines = []
summary_lines.append("# Final Reviewer Run Summary\n")
summary_lines.append(f"Output folder: `{OUT}`\n")
summary_lines.append(f"RMSE source mode: `{source_mode}`\n")
if source_mode == "aggregate_only_fallback":
    summary_lines.append("**WARNING:** Tables/figures are generated from aggregate draft numbers, not full per-window predictions. This is okay for revising the draft, but not a full model rerun.\n")

summary_lines.append("## Main RMSE tables\n")
for dataset in main["dataset"].unique():
    piv = main[main["dataset"]==dataset].pivot_table(index="method", columns="task_horizon", values="rmse", aggfunc="first")
    cols = [c for c in HORIZON_ORDER if c in piv.columns] + [c for c in piv.columns if c not in HORIZON_ORDER]
    idx = [m for m in PAPER_METHOD_ORDER if m in piv.index] + [m for m in piv.index if m not in PAPER_METHOD_ORDER]
    piv = piv.loc[idx, cols]
    summary_lines.append(f"### {dataset}\n")
    try:
        summary_lines.append(piv.round(3).to_markdown())
    except Exception:
        summary_lines.append(str(piv.round(3)))
    summary_lines.append("\n")

summary_lines.append("## Paired bootstrap MNAR - MAR deltas\n")
try:
    b2 = boot.drop(columns=[c for c in ["horizon_order"] if c in boot.columns])
    summary_lines.append(b2.round(4).to_markdown(index=False))
except Exception:
    summary_lines.append(str(boot.round(4)))
summary_lines.append("\n")

summary_lines.append("## Reviewer checklist\n")
checklist = {
    "Main RMSE long table": TABLE_DIR / "main_rmse_long.csv",
    "Seattle RMSE wide table": TABLE_DIR / "seattle_rmse_wide.csv",
    "METR-LA RMSE wide table": TABLE_DIR / "metrla_rmse_wide.csv",
    "Paired MNAR-MAR bootstrap table": TABLE_DIR / "paired_bootstrap_mnar_minus_mar.csv",
    "Imputation RMSE figure": FIG_DIR / "impute_rmse_methods_ci_labels.png",
    "Forecast horizon figure": FIG_DIR / "forecast_rmse_by_horizon_panels_ci.png",
    "Synthetic alpha figure": FIG_DIR / "synthetic_validation_alpha_sweep.png",
    "Synthetic alpha table": TABLE_DIR / "synthetic_alpha_sweep_delta.csv",
    "Evaluation-window length counts": TABLE_DIR / "evaluation_window_length_bucket_counts.csv",
    "Panel missingness summary": TABLE_DIR / "panel_missingness_summary.csv",
    "Seed robustness summary if real seeds exist": TABLE_DIR / "seed_robustness_summary.csv",
}
for name, path in checklist.items():
    summary_lines.append(f"- [{'x' if path.exists() else ' '}] {name}: `{path}`")

summary_path = OUT / "paper_numbers_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))

# LaTeX preview for bootstrap table
latex_boot = boot.copy()
latex_boot = latex_boot[latex_boot["dataset"].eq("Seattle")]
if not latex_boot.empty:
    latex_boot["Delta RMSE"] = latex_boot["delta_rmse"].map(lambda x: f"{x:.3f}")
    latex_boot["95% CI"] = latex_boot.apply(lambda r: f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
    latex_small = latex_boot[["task_horizon", "Delta RMSE", "95% CI"]].rename(columns={"task_horizon":"Task"})
    (TABLE_DIR / "latex_bootstrap_table_preview.txt").write_text(
        latex_small.to_latex(index=False, escape=False),
        encoding="utf-8"
    )
    print("Saved LaTeX bootstrap preview:", TABLE_DIR / "latex_bootstrap_table_preview.txt")



## What to do with this output

Use:

```text
final_reviewer_outputs/paper_numbers_summary.md
```

as the source of truth for the draft.

If the summary says `aggregate_only_fallback`, then do **not** claim this notebook performed a fresh full training rerun. Say the paper tables were regenerated from the final aggregate numbers, and only claim paired-bootstrap/seed results if those files come from the real per-window runs.
